# Step 5: Vectorized Backtesting Engine

This notebook demonstrates the execution of the modular, strategy-independent **Vectorized Backtesting Engine**.

### Objective
Load generated signals for both **Momentum (Trend Crossover)** and **Mean Reversion (Rolling Z-Score)** strategies, simulate execution at tomorrow's Open (`next_open` to prevent look-ahead bias), apply transaction costs (5 bps) and slippage (2 bps), and simulate daily portfolio balances vectorially.

## 1. Engine Architecture & Mathematical Formulations

The engine decomposes backtesting into discrete functions:
1. **Execution Model**: Maps daily signals to next-day positions (1-day lag for Open execution, 2-day lag for Close execution).
2. **Returns Calculation**: Computes simple or log returns. For `next_open` execution, intraday returns are captured on entry, close-to-close returns on hold days, and overnight returns on exit days.
3. **Costs & Slippage**: Charged only when positions change. A sign change (long-to-short flip) is decomposed into an exit and an entry, charging cost on both.
4. **Portfolio Simulation**: Simulates cumulative growth and rupee equity values.

In [ ]:
import os
import sys

# Insert project source root to system path for local imports
sys.path.append(os.path.abspath('../'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.engine import backtest
from src.validators import validate_backtest_inputs, validate_backtest_results

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Setup complete. Modules imported successfully.")

## 2. Load Price and Signal Data

We load preprocessed index close/open prices and the signals generated in Step 4A (Momentum) and Step 4B (Mean Reversion).

In [ ]:
prices_path = "../data/processed/nifty50_clean.parquet"
mom_signals_path = "../data/processed/momentum_signals.parquet"
mr_signals_path = "../data/processed/mean_reversion_signals.parquet"

prices_df = pd.read_parquet(prices_path)
mom_df = pd.read_parquet(mom_signals_path)
mr_df = pd.read_parquet(mr_signals_path)

print(f"Prices shape: {prices_df.shape}")
print(f"Momentum signals shape: {mom_df.shape}")
print(f"Mean Reversion signals shape: {mr_df.shape}")

## 3. Backtest Momentum Strategy

We run the backtest engine on the Momentum crossover signals. We configure:
- Initial Capital: ₹100,000
- Transaction Cost: 5 bps (0.05%)
- Slippage: 2 bps (0.02%)
- Execution Type: `next_open`

In [ ]:
mom_result = backtest(
    prices=prices_df,
    signals=mom_df["Raw_Signal"],
    initial_capital=100000.0,
    transaction_cost_bps=5.0,
    slippage_bps=2.0,
    return_type="simple",
    execution_type="next_open"
)

print("=== MOMENTUM BACKTEST SUMMARY ===")
for k, v in mom_result.summary.items():
    if isinstance(v, float):
        print(f"{k:25}: {v:.4f}")
    else:
        print(f"{k:25}: {v}")

### Inspect Momentum Trade Book

We print the first 10 trades detected in the Momentum simulation.

In [ ]:
print(f"Total trades in Momentum book: {len(mom_result.trade_book)}")
display(mom_result.trade_book.head(10))

## 4. Backtest Mean Reversion Strategy

We run the backtest engine on the Mean Reversion Z-score signals using the same parameters.

In [ ]:
mr_result = backtest(
    prices=prices_df,
    signals=mr_df["Raw_Signal"],
    initial_capital=100000.0,
    transaction_cost_bps=5.0,
    slippage_bps=2.0,
    return_type="simple",
    execution_type="next_open"
)

print("=== MEAN REVERSION BACKTEST SUMMARY ===")
for k, v in mr_result.summary.items():
    if isinstance(v, float):
        print(f"{k:25}: {v:.4f}")
    else:
        print(f"{k:25}: {v}")

### Inspect Mean Reversion Trade Book

We print the first 10 trades detected in the Mean Reversion simulation.

In [ ]:
print(f"Total trades in Mean Reversion book: {len(mr_result.trade_book)}")
display(mr_result.trade_book.head(10))

## 5. Visualizing Equity Curves & Trade Growth

We plot the simulated equity curves of both strategies to compare performance visually.

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(mom_result.equity_curve.index, mom_result.equity_curve, label=f"Momentum Crossover (Final: ₹{mom_result.summary['Final_Capital']:,.2f})", color="#1f77b4")
plt.plot(mr_result.equity_curve.index, mr_result.equity_curve, label=f"Mean Reversion Z-Score (Final: ₹{mr_result.summary['Final_Capital']:,.2f})", color="#ff7f0e")
plt.axhline(100000.0, color="gray", linestyle="--", alpha=0.7, label="Initial Capital (₹100,000)")
plt.title("Nifty 50 Strategy Equity Curves Comparison (₹100k Initial Capital)", fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Portfolio Value (₹)")
plt.legend(loc="upper left")
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

## 6. Export Results

We save the results for both strategies to data/processed for subsequent performance metrics evaluation.

In [ ]:
# Save Momentum
mom_result.portfolio.to_parquet("../data/processed/momentum_portfolio_history.parquet")
mom_result.trade_book.to_csv("../data/processed/momentum_trade_book.csv", index=False)

# Save Mean Reversion
mr_result.portfolio.to_parquet("../data/processed/mean_reversion_portfolio_history.parquet")
mr_result.trade_book.to_csv("../data/processed/mean_reversion_trade_book.csv", index=False)

print("Backtest simulation results exported successfully.")